# Corrective RAG (CRAG) with LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/17-corrective-rag/corrective_rag_workbook.ipynb)

---

## What is Corrective RAG?

Standard RAG retrieves documents and generates an answer — no matter how relevant (or irrelevant) those documents are. **Corrective RAG (CRAG)** adds a self-correction loop:

1. **Retrieve** — Fetch top-k documents from a vectorstore
2. **Grade** — An LLM evaluates each document: relevant (`yes`) or not (`no`)
3. **Correct (if needed)** — If any document is irrelevant, rewrite the query and fall back to web search
4. **Generate** — Answer using only the high-quality context

This workbook implements the full CRAG pipeline from [Yan et al. (2024)](https://arxiv.org/abs/2401.15884) using **LangGraph 0.6.x**.

### Graph architecture

```
START
  |
retrieve          <- Chroma vectorstore, top-3 docs
  |
grade_documents   <- LLM scores each doc yes/no
  |
  +- all relevant ---------------------------+
  |                                          |
  +- any irrelevant -> transform_query       |
                           |                 |
                     web_search_node         |
                           |                 |
                         generate <----------+
                           |
                          END
```

### What you will learn

- How to use `with_structured_output` for LLM grading
- How to wire conditional edges in LangGraph
- How query rewriting improves web search recall
- How to stream LangGraph state updates node-by-node

## Cell 1 — Install dependencies

Only runs on Google Colab. Skip locally if you already have the packages.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q \
        langchain==0.3.27 \
        langchain-core==0.3.79 \
        langchain-openai==0.3.33 \
        langchain-chroma==0.2.6 \
        langchain-community==0.3.31 \
        langchain-text-splitters==0.3.11 \
        langgraph==0.6.7 \
        duckduckgo-search==8.1.1 \
        chromadb==1.1.0 \
        python-dotenv
    print("Dependencies installed.")
else:
    print("Local environment -- skipping install.")

## Cell 2 — API key

On Colab: enter your key when prompted. Locally: reads from `.env`.

In [ ]:
import os

if "google.colab" in sys.modules:
    import getpass

    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
else:
    from dotenv import load_dotenv

    load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set!"
print("API key loaded.")

## Cell 3 — Build the knowledge base

We create a small in-memory Chroma vectorstore from five documents covering LangGraph, CRAG, and RAG concepts. This simulates a domain-specific corpus your students might build from their own PDFs or URLs.

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain with the ability to coordinate multiple chains in a graph of nodes and edges, supporting cycles, branching, and checkpointing."
    ),
    Document(
        page_content="Corrective RAG (CRAG) improves retrieval by grading each retrieved document for relevance. Irrelevant documents trigger a query rewrite followed by a web search, replacing low-quality local context with fresh information."
    ),
    Document(
        page_content="RAG (Retrieval Augmented Generation) grounds LLM responses in a knowledge base. A retriever fetches relevant documents; those documents become the context an LLM uses to generate a factual, sourced answer."
    ),
    Document(
        page_content="LangGraph StateGraph lets you define typed state, add nodes as Python functions, and wire them with edges and conditional edges. The compiled graph handles state transitions, streaming, and optional persistence via checkpointers."
    ),
    Document(
        page_content="ChromaDB is an open-source embedding database. LangChain's Chroma integration stores document embeddings locally and supports cosine-similarity retrieval via the as_retriever() interface."
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(CORPUS)

vectorstore = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="crag-workbook",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vectorstore ready. {len(chunks)} chunks indexed.")

## Cell 4 — Define the LLM and grader

The **grader** uses `with_structured_output` to force the LLM to return a Pydantic model with a `score` field of either `'yes'` or `'no'`. This is the modern LangChain pattern for structured LLM output -- no parsing, no regex.

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel


class GradeDocuments(BaseModel):
    score: Literal["yes", "no"]


llm = ChatOpenAI(model="gpt-5-nano")
grader = llm.with_structured_output(GradeDocuments)

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Is this document relevant to the question? Reply 'yes' or 'no' only."),
        ("human", "Question: {question}\n\nDocument:\n{document}"),
    ]
)

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer using only the context below. Be concise.\n\nContext:\n{context}"),
        ("human", "{question}"),
    ]
)

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Rewrite the question to make it more specific and web-searchable."),
        ("human", "{question}"),
    ]
)

print("LLM and grader ready.")

## Cell 5 — Quick grader smoke test

Before building the full graph, verify the grader works on its own.

In [ ]:
test_q = "What is Corrective RAG?"

test_cases = [
    (
        "Relevant doc",
        "CRAG improves retrieval by grading each document for relevance before generating.",
    ),
    ("Irrelevant doc", "The 2024 Olympics took place in Paris, France."),
]

for label, doc_text in test_cases:
    result = (grade_prompt | grader).invoke({"question": test_q, "document": doc_text})
    print(f"{label}: score = {result.score!r}")

## Cell 6 — Build the CRAG graph

Key LangGraph patterns used here:

| Pattern | Where | Purpose |
|---|---|---|
| `TypedDict` state | `CRAGState` | Typed, mutable state bag passed to every node |
| Plain node functions | `retrieve`, `grade_documents`, ... | Each returns a partial state update dict |
| `add_conditional_edges` | after `grade_documents` | Routes to web path or direct generate |
| Plain `list` (no reducer) | `documents` field | Each node explicitly replaces the list |


In [ ]:
from typing import TypedDict

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.documents import Document
from langgraph.graph import END, START, StateGraph

web_search = DuckDuckGoSearchResults(max_results=3)


class CRAGState(TypedDict):
    question: str
    documents: list
    generation: str
    web_search: bool


def retrieve(state: CRAGState) -> dict:
    docs = retriever.invoke(state["question"])
    return {"documents": docs, "web_search": False}


def grade_documents(state: CRAGState) -> dict:
    relevant, needs_web = [], False
    for doc in state["documents"]:
        result = (grade_prompt | grader).invoke(
            {
                "question": state["question"],
                "document": doc.page_content,
            }
        )
        if result.score == "yes":
            relevant.append(doc)
        else:
            needs_web = True
    return {"documents": relevant, "web_search": needs_web}


def transform_query(state: CRAGState) -> dict:
    rewritten = (rewrite_prompt | llm).invoke({"question": state["question"]})
    return {"question": rewritten.content}


def web_search_node(state: CRAGState) -> dict:
    results = web_search.invoke(state["question"])
    web_docs = [Document(page_content=str(results))]
    return {"documents": state["documents"] + web_docs}


def generate(state: CRAGState) -> dict:
    context = "\n\n".join(d.page_content for d in state["documents"])
    if not context:
        return {"generation": "I do not have enough information to answer this question."}
    response = (answer_prompt | llm).invoke({"context": context, "question": state["question"]})
    return {"generation": response.content}


def decide_after_grading(state: CRAGState) -> Literal["transform_query", "generate"]:
    return "transform_query" if state["web_search"] else "generate"


graph = StateGraph(CRAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("grade_documents", grade_documents)
graph.add_node("transform_query", transform_query)
graph.add_node("web_search_node", web_search_node)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "grade_documents")
graph.add_conditional_edges("grade_documents", decide_after_grading)
graph.add_edge("transform_query", "web_search_node")
graph.add_edge("web_search_node", "generate")
graph.add_edge("generate", END)

app = graph.compile()
print("CRAG graph compiled.")

## Cell 7 — Visualise the graph

In [ ]:
try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph render unavailable: {e}")
    print(app.get_graph().draw_mermaid())

## Cell 8 — Run with in-corpus questions

These questions are covered by the knowledge base. Expect `web_search: False`.

In [ ]:
INIT = lambda q: {"question": q, "documents": [], "generation": "", "web_search": False}

in_corpus = [
    "What is Corrective RAG and how does it improve retrieval?",
    "How does LangGraph StateGraph handle conditional branching?",
    "What is ChromaDB and how does it integrate with LangChain?",
]

for q in in_corpus:
    r = app.invoke(INIT(q))
    print(f"Q: {q}")
    print(f"   Web search: {r['web_search']}  |  Docs: {len(r['documents'])}")
    print(f"   A: {r['generation']}")
    print()

## Cell 9 — Run with out-of-corpus questions

These questions are not in the knowledge base. Expect `web_search: True` and a rewritten query.

In [ ]:
out_of_corpus = [
    "Who won the 2024 US presidential election?",
    "What is the latest Python version as of 2025?",
]

for q in out_of_corpus:
    r = app.invoke(INIT(q))
    print(f"Q: {q}")
    print(f"   Web search: {r['web_search']}  |  Docs: {len(r['documents'])}")
    print(f"   A: {r['generation']}")
    print()

## Cell 10 — Stream execution for observability

LangGraph's `.stream(stream_mode='updates')` emits one dict per node as it finishes. You can watch every step of the CRAG pipeline unfold.

In [ ]:
question = "What are the key differences between CRAG and standard RAG?"
print(f'Streaming CRAG for: "{question}"\n')

for step in app.stream(INIT(question), stream_mode="updates"):
    node_name = list(step.keys())[0]
    out = step[node_name]
    print(f"[{node_name}]")
    if "documents" in out:
        print(f"  docs kept: {len(out['documents'])}")
    if "web_search" in out:
        print(f"  web_search flag: {out['web_search']}")
    if "question" in out:
        print(f"  rewritten question: {out['question']}")
    if "generation" in out:
        print(f"  answer: {out['generation']}")
    print()

## Exercises

### Exercise 1 — Stricter grader
Modify `grade_prompt` to require that a document contain **specific facts** rather than just topic overlap. Re-run Cell 8 and observe which documents now get filtered.

### Exercise 2 — Confidence threshold
Extend `GradeDocuments` with a `confidence: float` field (0.0 to 1.0). Only keep documents with `confidence >= 0.7`. Update both the Pydantic model and the `grade_documents` node.

### Exercise 3 — Expand the corpus
Add five new documents to `CORPUS` on a topic of your choice (e.g. ReAct agents, LangChain tools, vector databases). Rebuild the vectorstore in Cell 3 and verify those topics no longer trigger web search.

### Exercise 4 — Persistent checkpointing
Recompile `app` with an in-memory checkpointer:
```python
from langgraph.checkpoint.memory import MemorySaver
app = graph.compile(checkpointer=MemorySaver())
```
Invoke with `config={'configurable': {'thread_id': 'demo'}}`. Inspect state after each run with `app.get_state(config)`.

---

## What's next?

- **Example 18 — Self-Reflecting Agent** — Reflexion loop: generate, critique, revise until confident
- **Example 22 — Hybrid Search RAG** — BM25 + vector similarity via EnsembleRetriever
- **LangGraph docs** — https://langchain-ai.github.io/langgraph/
- **CRAG paper** — Yan et al. (2024), arXiv:2401.15884